# Ground Truth C-Space: Generation & Visualization
This notebook handles both the **Generation** of the 3D ground truth C-Space (via brute-force collision checking in MuJoCo) and its **Interactive Visualization**.

In [ ]:
import os
import numpy as np
import mujoco
import plotly.graph_objects as go
from skimage import measure
import plotly.io as pio
from tqdm.auto import tqdm
from multiprocessing import Pool
from jaco_arm.jaco import JacoEnv

pio.renderers.default = 'iframe_connected' # Optimized for local Jupyter/VS Code

Imports successful.


## 1. Data Generation Logic
We use a parallelized oracle to check collisions on a high-resolution grid.

In [ ]:
_GLOBAL_ENV = None

def _init_oracle(model_path):
    global _GLOBAL_ENV
    # Standard resolution for oracle checks
    _GLOBAL_ENV = JacoEnv(width=101, height=101, model_path=model_path)

def _oracle_worker(q):
    env = _GLOBAL_ENV
    # Check Joint Limits
    for i in range(len(q)):
        if env.model.jnt_limited[i]:
            lo, hi = env.model.jnt_range[i]
            if q[i] < lo - 1e-4 or q[i] > hi + 1e-4:
                return 1 # Collision
        env.data.qpos[i] = q[i]
    
    mujoco.mj_forward(env.model, env.data)
    
    # Check Physical Contacts
    for i in range(env.data.ncon):
        c = env.data.contact[i]
        g1, g2 = c.geom1, c.geom2
        fixed_geoms = {0, 1, 2, 12}
        if g1 in fixed_geoms and g2 in fixed_geoms: continue
        b1, b2 = env.model.geom_bodyid[g1], env.model.geom_bodyid[g2]
        if env.model.body_parentid[b1] == b2 or env.model.body_parentid[b2] == b1: continue
        return 1 # Collision
    return -1 # Free

def get_true_cspace(res=64, force=False):
    out_dir = "output/ndf"
    grid_path = os.path.join(out_dir, "jaco_cspace.npy")
    
    if os.path.exists(grid_path) and not force:
        grid = np.load(grid_path)
        ticks = [np.load(os.path.join(out_dir, f"jaco_theta{i}.npy")) for i in range(3)]
        return grid, ticks

    model_path = "jaco_3dof.xml"
    n_workers = os.cpu_count() or 4
    ticks = np.linspace(-np.pi, np.pi, res)
    grid_points = np.stack(np.meshgrid(ticks, ticks, ticks, indexing='ij'), axis=-1).reshape(-1, 3)

    with Pool(processes=n_workers, initializer=_init_oracle, initargs=(model_path,)) as pool:
        results = list(tqdm(pool.imap(_oracle_worker, grid_points, chunksize=512), total=len(grid_points)))
    
    dist_values = np.array(results).reshape(res, res, res)
    grid = (dist_values == 1).astype(int)
    
    os.makedirs(out_dir, exist_ok=True)
    np.save(grid_path, grid)
    for i in range(3):
        np.save(os.path.join(out_dir, f"jaco_theta{i}.npy"), ticks)
    
    return grid, [ticks]*3

In [3]:
grid, ticks = get_true_cspace(res=64, force=False)
print(f"Final Grid Shape: {grid.shape}")

Loading existing C-space from output/ndf/jaco_cspace.npy
Final Grid Shape: (64, 64, 64)


## 2. Visualization
Extract isosurface and plot with Plotly.

In [ ]:
verts, faces, normals, values = measure.marching_cubes(grid, level=0.5)

for i in range(3):
    verts[:, i] = np.interp(verts[:, i], np.arange(len(ticks[i])), ticks[i])

Extracting isosurface...
Extracted mesh with 20151 vertices and 40298 faces.


In [5]:
fig = go.Figure(data=[
    go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color='blue', opacity=0.6, name='Ground Truth'
    )
])

fig.update_layout(
    title="Ground Truth C-Space Mesh (Interactive)",
    scene=dict(
        xaxis=dict(title='θ₁', tickvals=[-np.pi, 0, np.pi], ticktext=['-π', '0', 'π']),
        yaxis=dict(title='θ₂', tickvals=[-np.pi, 0, np.pi], ticktext=['-π', '0', 'π']),
        zaxis=dict(title='θ₃', tickvals=[-np.pi, 0, np.pi], ticktext=['-π', '0', 'π']),
        aspectmode='cube'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

html_path = 'output/ndf/visualize_gt.html'
fig.write_html(html_path)
print(f"Interactive plot saved as fallback to {html_path}")

Interactive plot saved as fallback to output/ndf/visualize_gt.html
